<a href="https://colab.research.google.com/github/nattaponm/Teach_Nowcast_Pysteps_TMD/blob/main/notebooks/00_prepare_tmd_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 0: เตรียมข้อมูลเรดาร์ TMD ให้พร้อมใช้กับ pysteps
### From TMD CAPPI GeoTIFF to quality-controlled pysteps data

⏱️ เวลาเรียนโดยประมาณ **60 นาที** | 🌩️ กรณีศึกษา: **พายุลูกเห็บ อ.เชียงของ จ.เชียงราย 23 เม.ย. 2020**

---
## 🎯 วัตถุประสงค์การเรียนรู้
เมื่อจบ notebook นี้ นิสิตจะสามารถ
1. **อ่านข้อมูล raster เชิงภูมิศาสตร์ (GeoTIFF)** และอธิบาย CRS, affine transform และ nodata
2. **สร้าง `metadata`** ที่ pysteps ต้องการจากข้อมูล GeoTIFF ได้เอง
3. **ทำ quality control (QC)** ข้อมูลเรดาร์เบื้องต้น เช่น range mask, noise, speckle, radial spike และ ground clutter
4. **บันทึกข้อมูล** เป็น NetCDF 3 ชุดที่ใช้ใน Notebook 1–5

> 💡 notebook นี้ **นิสิตรันเองได้** แต่ใน repo มีไฟล์ผลลัพธ์ที่เตรียมไว้แล้วในโฟลเดอร์ `data/` ด้วย ถ้ารันไม่ผ่านก็ยังเรียน Notebook 1–5 ต่อได้

## 📖 1. แนวคิดพื้นฐาน

### 1.1 CAPPI คืออะไร
เรดาร์ตรวจอากาศหมุนสแกนหลายมุมเงย (elevation angle) ได้ข้อมูลรูปกรวยเรียกว่า **PPI** (Plan Position Indicator) แต่ละ PPI จะสูงขึ้นตามระยะทาง (ไกลเรดาร์ = ลำคลื่นอยู่สูง)
**CAPPI** (Constant Altitude PPI) คือการประมาณค่าจากหลาย PPI ให้ได้ภาพที่ **ความสูงคงที่** ในที่นี้คือ **2 km** เหนือระดับเรดาร์ จึงเปรียบเทียบพื้นที่ใกล้และไกลเรดาร์ได้ดีกว่า PPI

```
 ความสูง
   ▲        PPI มุมสูง  ╱
   │              ╱  ╱   PPI มุมกลาง
 2 km ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─   ← CAPPI 2 km
   │      ╱  ╱  ╱ ___----  PPI มุมต่ำ
   │  ╱ ╱ ___---
   📡────────────────────────────►  ระยะจากเรดาร์
```
ข้อจำกัด: ที่ระยะไกลมาก แม้ PPI มุมต่ำสุดก็อยู่สูงกว่า 2 km แล้ว ทำให้ CAPPI 2 km มี **รัศมีใช้งานจำกัด** (ซึ่งเราจะหาจากข้อมูลจริงในข้อ 5)

### 1.2 GeoTIFF และ affine transform
GeoTIFF คือภาพ raster ที่มีข้อมูลพิกัดกำกับ ตำแหน่งจุดกึ่งกลางของ pixel แถว $r$ คอลัมน์ $c$ คำนวณจาก **affine transform**

$$
x = x_{0} + \left(c + \tfrac{1}{2}\right)\Delta x, \qquad
y = y_{0} - \left(r + \tfrac{1}{2}\right)\Delta y
$$

โดย $(x_0, y_0)$ คือมุมบนซ้ายของภาพ และ $\Delta x, \Delta y$ คือขนาด pixel ข้อมูลชุดนี้ใช้ระบบพิกัด **UTM zone 47N (EPSG:32647)** หน่วยเมตร

### 1.3 สิ่งรบกวน (artifacts) ในข้อมูลเรดาร์
| ประเภท | ลักษณะในภาพ | วิธีจัดการใน notebook นี้ |
|---|---|---|
| **Noise** | จุดค่าต่ำกระจายทั่ว | ตัดค่าที่ต่ำกว่า threshold (เช่น 15 dBZ) |
| **Speckle** | จุดเดี่ยว ๆ ขนาดเล็กมาก | ลบกลุ่ม pixel ที่เล็กกว่า N pixel |
| **Radial spike / interference** | เส้นตรงพุ่งออกจากเรดาร์ (เช่น สัญญาณ RLAN 5 GHz) | ลบวัตถุที่ยาว แคบ และวางตัวตามแนวรัศมี |
| **Ground clutter** | echo จากภูเขา/สิ่งก่อสร้าง ซึ่ง **อยู่กับที่** และมีค่าเกือบคงที่ทุกภาพ | หา pixel ที่ "เปียก" เกือบทุกภาพและค่าแปรผันน้อย แล้วลบออก |
| **Beam blockage, attenuation** | เงาบังเป็นรูปพัด, ความเข้มลดลงหลังพายุแรง | ต้องใช้ข้อมูล polar / dual-pol (อยู่นอกขอบเขตของวิชานี้) |

### 1.4 เวลาในข้อมูล
ชื่อไฟล์ `cappi_2km_202004231100.tif` คือเวลา **11:00 UTC** ในที่นี้ **สันนิษฐานว่าเป็น UTC** (= 18:00 น. เวลาประเทศไทย) ซึ่งสอดคล้องกับข่าวพายุลูกเห็บที่เชียงของที่รายงานในวันที่ 24 เม.ย. 2563

## ⚙️ 2. เตรียมสภาพแวดล้อม

In [ ]:
#@title ⚙️ Setup: ติดตั้ง pysteps และดาวน์โหลดข้อมูล TMD (รันก่อนเสมอ ~2-3 นาทีใน Colab)
import os, sys, warnings
warnings.filterwarnings("ignore")

REPO = "Teach_Nowcast_Pysteps_TMD"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !pip install -q pysteps==1.21.5 cartopy netCDF4 rasterio geopandas
    if not os.path.exists(REPO):
        !git clone -q --depth 1 https://github.com/nattaponm/{REPO}.git

# หาโฟลเดอร์หลักของ repo (ที่มี utils/ และ data/)
for ROOT in [REPO, ".", ".."]:
    if os.path.exists(os.path.join(ROOT, "utils", "tmd_tools.py")):
        break
ROOT = os.path.abspath(ROOT)
sys.path.insert(0, ROOT)
DATA = os.path.join(ROOT, "data")

import numpy as np
import matplotlib.pyplot as plt
import pysteps
from utils.data_io import load_radar_nc
from utils import mapplot as mp
from utils import tmd_tools as tt
mp.set_style()
mp.BASEMAP.update(provinces=True, rivers=True, lakes=False)   # จังหวัด + แม่น้ำโขง
mp.POINTS[:] = tt.POIS                                         # เรดาร์ เชียงของ เชียงแสน
print("✅ พร้อมใช้งาน | ข้อมูลอยู่ที่:", DATA)

## 💻 3. รายการไฟล์และเวลา

In [ ]:
import glob, re, datetime as dt
import rasterio

RAW = os.path.join(ROOT, "0data")
files = sorted(glob.glob(os.path.join(RAW, "cappi_2km_*.tif")))

def time_from_name(path):
    """ดึงเวลา YYYYMMDDHHMM จากชื่อไฟล์"""
    s = re.search(r"(\d{12})", os.path.basename(path)).group(1)
    return dt.datetime.strptime(s, "%Y%m%d%H%M")

timestamps = np.array([time_from_name(f) for f in files])
print("จำนวนไฟล์:", len(files))
for f, t in zip(files[:3], timestamps[:3]):
    print(f"  {os.path.basename(f)} → {mp.tlabel(t)}")
print("  ...")
print("ช่วงเวลาห่างกัน:", sorted(set(np.diff(timestamps))))

## 🗺️ 4. อ่าน GeoTIFF และสร้าง metadata ของ pysteps

In [ ]:
with rasterio.open(files[0]) as src:
    print("CRS        :", src.crs)
    print("ขนาด (y,x) :", src.height, src.width)
    print("pixel size :", src.res, "m")
    print("bounds     :", src.bounds)
    print("nodata     :", src.nodata)
    print("transform  :\n", src.transform)
    crs, bounds, res = src.crs, src.bounds, src.res

Z_raw = np.stack([rasterio.open(f).read(1) for f in files]).astype("float64")
print("\nradar cube (time, y, x):", Z_raw.shape,
      f"| ค่า dBZ: {np.nanmin(Z_raw):.1f} ถึง {np.nanmax(Z_raw):.1f}",
      f"| NaN {100*np.isnan(Z_raw).mean():.1f} %")

**สร้าง `metadata`** ซึ่งเป็น dict ที่ pysteps ใช้กำกับทุก array (เทียบกับ Notebook 1 ของชุดข้อมูลตัวอย่าง)
- `projection` ต้องอยู่ในรูป **proj4 string** จึงแปลงจาก EPSG ด้วย `pyproj`
- `x1, y1` คือมุม **ล่างซ้าย** ส่วน `x2, y2` คือมุม **บนขวา** และ `yorigin="upper"` หมายถึงแถวแรกอยู่ด้านเหนือ (เหมือน GeoTIFF)

In [ ]:
from pyproj import CRS

metadata = {
    "projection": CRS.from_user_input(crs).to_proj4(),
    "x1": bounds.left, "x2": bounds.right,
    "y1": bounds.bottom, "y2": bounds.top,
    "xpixelsize": res[0], "ypixelsize": res[1],
    "cartesian_unit": "m", "yorigin": "upper",
    "unit": "dBZ", "transform": "dB",
    "accutime": 15.0,                    # นาทีระหว่างภาพ
    "zerovalue": 0.0, "threshold": 15.0,
    "institution": "Thai Meteorological Department (TMD)",
    "timestamps": timestamps,
}
for k, v in metadata.items():
    if k != "timestamps":
        print(f"{k:15s}: {v}")

### แผนที่ข้อมูลดิบ (ก่อน QC)

In [ ]:
i13 = 8                                   # 13:00 UTC — ภาพที่มี artifact ชัดที่สุด
BOX = tt.range_box(metadata, 205)         # ซูมให้เห็นรัศมี 200 km
fig, axs = mp.map_panels(1, 2, metadata, panel_width=4.0, zoom=BOX)
for j, i in enumerate([2, i13]):
    im = mp.radar_map(axs[j], Z_raw[i], metadata, units="dBZ", title=f"Raw CAPPI 2 km, {mp.tlabel(timestamps[i])}",
                      gridline_labels=("left", "bottom") if j == 0 else ("bottom",), north=(j == 0), zoom=BOX)
    mp.add_range_rings(axs[j], metadata, radii_km=(50, 100, 150, 200))
mp.add_colorbar(fig, im, axs)
mp.panel_labels(axs)
mp.save_figure(fig, "nb0_fig01_raw_cappi")
plt.show()

> 🔍 **สังเกต:** (1) พื้นที่สีเทาคือ NaN ซึ่งรวมทั้ง **"ไม่มี echo"** และ **"นอกรัศมีเรดาร์"** ไว้ด้วยกัน (2) ภาพ (b) มี **เส้นพุ่งออกจากเรดาร์เป็นรูปพัด** ทางทิศใต้–ตะวันตกเฉียงใต้ ซึ่งไม่ใช่ฝนจริง

## 🧹 5. Quality control (QC) ทีละขั้น

### ขั้นที่ 1: Range mask แยก "ไม่มีฝน" ออกจาก "ไม่มีข้อมูล"
pysteps ต้องการให้ **0 = ไม่มีฝน** และ **NaN = ไม่มีข้อมูล** เราจึงหารัศมีไกลสุดที่เรดาร์ตรวจพบ echo จากทุกภาพ แล้วกำหนดเป็นรัศมีใช้งาน

In [ ]:
rng_km = tt.distance_to_radar_km(metadata)
has_echo = np.isfinite(Z_raw).any(axis=0)
print(f"echo ไกลสุด: {rng_km[has_echo].max():.1f} km | percentile 99.9: {np.quantile(rng_km[has_echo], 0.999):.1f} km")

MAX_RANGE_KM = 160.0
range_mask = rng_km <= MAX_RANGE_KM
print(f"ใช้รัศมี {MAX_RANGE_KM:.0f} km → ครอบคลุม {100*range_mask.mean():.1f} % ของกริด")

### ขั้นที่ 2–4: noise, speckle และ radial spike
การตรวจหา **radial spike** ใช้หลัก *object-based* โดยมองแต่ละกลุ่ม pixel ที่ติดกันเป็นวัตถุ 1 ชิ้น (ด้วย `skimage.measure.regionprops`) แล้ววัด
- **ความยาว/ความกว้าง (elongation)** $= L_{major}/L_{minor}$ ซึ่ง spike จะยาวและแคบมาก
- **มุมระหว่างแกนหลักของวัตถุกับแนวรัศมีจากเรดาร์** $\theta$ ซึ่ง spike จะชี้เข้าหาเรดาร์ ($\theta \approx 0$)
- **ค่า dBZ เฉลี่ย** ซึ่ง interference มักมีค่าต่ำถึงปานกลาง

วัตถุที่ elongation $\ge 4$, $\theta \le 15^\circ$, ยาวเกิน 10 km และ dBZ เฉลี่ย $< 35$ จะถูกลบ

In [ ]:
from skimage.measure import label, regionprops

def qc_frame(z, noise_dbz=15.0, min_area=8, elong_min=4.0, angle_tol=15.0,
             spike_max_dbz=35.0, min_length_px=20):
    """QC หนึ่งภาพ: คืนค่า (z_qc, ธง) โดย ธง = 0 ปกติ, 1 noise, 2 speckle, 3 spike"""
    flag = np.zeros(z.shape, dtype=np.int8)
    z = z.copy()
    # (2) noise threshold
    noise = np.isfinite(z) & (z < noise_dbz)
    flag[noise] = 1
    z[noise] = np.nan
    # (3)+(4) object-based
    ny, nx = z.shape
    cy, cx = (ny - 1) / 2, (nx - 1) / 2
    lab = label(np.isfinite(z), connectivity=2)
    for p in regionprops(lab, intensity_image=np.nan_to_num(z)):
        sel = lab == p.label
        if p.area < min_area:
            flag[sel], z[sel] = 2, np.nan
            continue
        elong = p.axis_major_length / max(p.axis_minor_length, 1.0)
        r0, c0 = p.centroid
        radial = np.arctan2(-(r0 - cy), c0 - cx)          # ทิศจากเรดาร์ไปยังวัตถุ
        major = np.pi / 2 + p.orientation                  # ทิศของแกนหลักของวัตถุ
        theta = np.degrees(np.abs((major - radial + np.pi / 2) % np.pi - np.pi / 2))
        if (elong >= elong_min and theta <= angle_tol and p.axis_major_length > min_length_px
                and p.intensity_mean < spike_max_dbz):
            flag[sel], z[sel] = 3, np.nan
    # (1) range mask: ในรัศมี + ไม่มี echo = 0 dBZ (ไม่มีฝน), นอกรัศมี = NaN
    z = np.where(np.isfinite(z), z, 0.0)
    z[~range_mask] = np.nan
    return z, flag

Z_qc, flags = zip(*[qc_frame(z) for z in Z_raw])
Z_qc, flags = np.stack(Z_qc), np.stack(flags)
for name, v in [("noise", 1), ("speckle", 2), ("spike", 3)]:
    print(f"{name:8s}: ลบไป {np.sum(flags == v):7d} pixel")

In [ ]:
removed = np.where(flags[i13] > 0, Z_raw[i13], np.nan)
BOX = tt.range_box(metadata, 165)
fig, axs = mp.map_panels(1, 3, metadata, panel_width=3.0, zoom=BOX)
im = mp.radar_map(axs[0], Z_raw[i13], metadata, units="dBZ", title="Raw", zoom=BOX)
mp.radar_map(axs[1], Z_qc[i13], metadata, units="dBZ", title="After QC",
             gridline_labels=("bottom",), north=False, zoom=BOX, points=False)
mp.radar_map(axs[2], removed, metadata, units="dBZ", title="Removed by QC",
             gridline_labels=("bottom",), north=False, zoom=BOX, points=False)
mp.add_colorbar(fig, im, axs, label="Reflectivity (dBZ)")
mp.panel_labels(axs)
fig.suptitle(f"Quality control, {mp.tlabel(timestamps[i13])}", x=0.02, ha="left")
mp.save_figure(fig, "nb0_fig02_quality_control")
plt.show()

> ⚠️ **ข้อควรระวัง:** QC ทุกวิธีมีทั้ง *false positive* (ลบฝนจริง) และ *false negative* (เหลือ artifact บางส่วน) ค่าพารามิเตอร์ข้างต้นปรับมาสำหรับเคสนี้ ถ้านำไปใช้กับเคสอื่นต้องตรวจด้วยตาเสมอ ในงานวิจัยควรรายงานขั้นตอน QC ไว้ในส่วน *Methods*

### ขั้นที่ 5: Ground clutter (echo ที่อยู่กับที่)
พายุ **เคลื่อนที่** แต่ ground clutter จากภูเขา **อยู่กับที่** ดังนั้น pixel ที่มี echo เกือบทุกภาพ และค่า dBZ แทบไม่เปลี่ยน น่าจะเป็น clutter

$$
f_{\mathrm{wet}} = \frac{1}{T}\sum_{t=1}^{T} \mathbf{1}\left[\mathrm{dBZ}_t \ge 15\right] \ \ge 0.9
\qquad\text{และ}\qquad
\sigma_t(\mathrm{dBZ}) < 5\ \mathrm{dB}
$$

> ⚠️ ในงาน operational จะสร้าง **clutter map** จากวันที่ท้องฟ้าแจ่มใส (clear-air) แต่ที่นี่มีข้อมูลเพียง 3 ชั่วโมง จึงใช้ข้อมูลของเหตุการณ์เอง ซึ่งอาจลบฝนจริงที่ตกค้างอยู่กับที่ไปด้วย

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    echo = np.where(Z_qc >= 15, Z_qc, np.nan)
    wet_frac = np.mean(Z_qc >= 15, axis=0)
    dbz_std = np.nanstd(echo, axis=0)
clutter = (wet_frac >= 0.9) & (dbz_std < 5.0)
print(f"clutter pixel: {clutter.sum()} ({clutter.sum() * (metadata['xpixelsize']/1000)**2:.0f} km²)")

fig, axs = mp.map_panels(1, 2, metadata, panel_width=3.8, zoom=BOX)
im = mp.field_map(axs[0], np.where(range_mask, wet_frac, np.nan), metadata, cmap="viridis",
                  levels=np.linspace(0, 1, 11), label="Fraction of scans with echo ≥ 15 dBZ",
                  title="Echo persistence", zoom=BOX)
mp.add_colorbar(fig, im, axs[0], shrink=0.9)
im2 = mp.radar_map(axs[1], np.where(clutter, np.nanmean(echo, axis=0), np.nan), metadata, units="dBZ",
                   title="Detected ground clutter (mean dBZ)", zoom=BOX,
                   gridline_labels=("bottom",), north=False)
mp.add_colorbar(fig, im2, axs[1], shrink=0.9)
mp.panel_labels(axs)
mp.save_figure(fig, "nb0_fig03_ground_clutter")
plt.show()

# ลบ clutter ทุกเวลา (ให้เป็น "ไม่มีฝน" = 0 dBZ) และบันทึกธง = 4
Z_qc[:, clutter] = 0.0
flags[:, clutter & np.isfinite(Z_raw).any(axis=0)] = 4

## 💾 6. สร้างและบันทึกข้อมูล 3 ชุด

| ไฟล์ | ตัวแปร | ความละเอียด | ใช้ใน |
|---|---|---|---|
| `tmd_cri_20200423_dbz_500m.nc` | reflectivity (dBZ) หลัง QC | 500 m | NB1 |
| `tmd_cri_20200423_dbz_1km.nc` | **ค่าสูงสุด** ของ dBZ ในช่อง 2×2 | 1 km | NB5 (T-DaTing ถูกปรับค่ามาสำหรับ 1 km) |
| `tmd_cri_20200423_rain_1km.nc` | rain rate (mm/h) **ค่าเฉลี่ย** ในช่อง 2×2 | 1 km | NB2–4 (nowcasting) |

**การแปลง dBZ → rain rate** ใช้ **Marshall–Palmer** ($Z = 200R^{1.6}$) และ **hail cap 55 dBZ** (ค่าที่สูงกว่านี้มักมาจากลูกเห็บ ถ้าแปลงตรง ๆ จะได้ฝนสูงเกินจริง) รายละเอียดอยู่ใน Notebook 1

> ❓ ทำไม dBZ ใช้ **ค่าสูงสุด** แต่ rain rate ใช้ **ค่าเฉลี่ย** เมื่อลดความละเอียด? เพราะ dBZ เป็นสเกลลอการิทึม การเฉลี่ยตรง ๆ ไม่ถูกต้องทางกายภาพ และ T-DaTing ถูกออกแบบมาสำหรับ *maximum reflectivity composite* ส่วน rain rate เป็นปริมาณเชิงเส้นที่เฉลี่ยได้

In [ ]:
from utils.data_io import save_radar_nc
from pysteps.utils.dimension import aggregate_fields_space

OUT = DATA
os.makedirs(OUT, exist_ok=True)
src_note = "TMD CAPPI 2 km (Chiang Rai radar), quality-controlled in 00_prepare_tmd_data.ipynb"

# (1) dBZ 500 m
meta_dbz = dict(metadata, source=src_note)
save_radar_nc(os.path.join(OUT, "tmd_cri_20200423_dbz_500m.nc"), Z_qc, meta_dbz)

# (2) dBZ 1 km = ค่าสูงสุดในช่อง 2×2
nt, ny, nx = Z_qc.shape
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    Z1 = np.nanmax(Z_qc.reshape(nt, ny // 2, 2, nx // 2, 2), axis=(2, 4))
meta_dbz1 = dict(meta_dbz, xpixelsize=2 * metadata["xpixelsize"], ypixelsize=2 * metadata["ypixelsize"])
save_radar_nc(os.path.join(OUT, "tmd_cri_20200423_dbz_1km.nc"), Z1, meta_dbz1)

# (3) rain rate 1 km (Marshall–Palmer + hail cap 55 dBZ) แล้วเฉลี่ย 2×2
R = tt.dbz_to_rainrate(Z_qc, a=200.0, b=1.6, hail_cap=55.0, min_dbz=15.0)
meta_r = dict(metadata, unit="mm/h", transform=None, threshold=0.1, zerovalue=0.0,
              zr_a=200.0, zr_b=1.6, source=src_note + "; Z=200R^1.6, hail cap 55 dBZ")
R1, meta_r1 = aggregate_fields_space(R, meta_r, (2 * metadata["xpixelsize"], 2 * metadata["ypixelsize"]))
save_radar_nc(os.path.join(OUT, "tmd_cri_20200423_rain_1km.nc"), R1, meta_r1)

for f in sorted(os.listdir(OUT)):
    if f.startswith("tmd_") and f.endswith(".nc"):
        p, m = load_radar_nc(os.path.join(OUT, f))
        print(f"{f:34s} {os.path.getsize(os.path.join(OUT, f))/1e6:5.1f} MB  shape={p.shape}  "
              f"unit={m['unit']:5s} max={np.nanmax(p):.1f}")

---
## 🧪 แบบฝึกหัด

### 🧪 แบบฝึกหัด 0.1: QC ลบอะไรไปบ้างในแต่ละเวลา?

นับจำนวน pixel ที่ถูกลบในแต่ละประเภท (noise = 1, speckle = 2, spike = 3, clutter = 4) **ทุกเวลา** (13 ภาพ) แล้วทำ **stacked bar chart** (แกน x = เวลา)

**คำถาม:** เวลาใดมี spike มากผิดปกติ? ถ้าเป็นผู้ดูแลเรดาร์ จะตรวจสอบอะไรต่อ?

<details>
<summary><b>💡 คำใบ้ (คลิกเพื่อเปิด)</b></summary>

- จำนวน spike ที่เวลา i: `np.sum(flags[i] == 3)`
- stacked bar: `ax.bar(x, a); ax.bar(x, b, bottom=a); ...`
- label เวลา: `[t.strftime("%H:%M") for t in timestamps]`

</details>

In [ ]:
# ✍️ แบบฝึกหัด 0.1 — เขียนโค้ดของคุณที่นี่ (ลบ # หน้าบรรทัดแล้วเติมส่วนที่เป็น ...)
# counts = {name: [np.sum(flags[i] == v) for i in range(len(timestamps))]
#           for name, v in [("noise", 1), ("speckle", 2), ("spike", 3), ("clutter", 4)]}
# fig, ax = plt.subplots(figsize=(7, 3))
# ...

In [ ]:
#@title ✅ เฉลยแบบฝึกหัด 0.1 (ลองทำเองก่อน! ดับเบิลคลิกเพื่อดูโค้ด แล้วกด ▶ เพื่อรัน)
labels_t = [t.strftime("%H:%M") for t in timestamps]
counts = {name: np.array([np.sum(flags[i] == v) for i in range(len(timestamps))])
          for name, v in [("noise", 1), ("speckle", 2), ("spike", 3), ("clutter", 4)]}
fig, ax = plt.subplots(figsize=(7, 3), layout="constrained")
bottom = np.zeros(len(timestamps))
for (name, c), col in zip(counts.items(), ["#bdbdbd", "#6baed6", "#e6550d", "#756bb1"]):
    ax.bar(labels_t, c / 1000, bottom=bottom, color=col, label=name)
    bottom += c / 1000
mp.skill_axes(ax, xlabel="Time (UTC)", ylabel="Removed pixels (×1000)", title="QC removals per scan")
ax.tick_params(axis="x", rotation=45)
ax.legend(frameon=False)
mp.save_figure(fig, "nb0_ex01_qc_counts")
plt.show()

### 🧪 แบบฝึกหัด 0.2: ผลของ noise threshold

เรียก `qc_frame(Z_raw[i13], noise_dbz=...)` ด้วยค่า **10, 15 และ 20 dBZ** แล้วทำแผนที่ 3 panel เปรียบเทียบ

**คำถาม:** ค่าใดเหมาะสมที่สุดสำหรับงาน (ก) nowcasting ฝนหนัก และ (ข) การศึกษาฝนเบา? มีข้อแลกเปลี่ยน (trade-off) อะไร?

<details>
<summary><b>💡 คำใบ้ (คลิกเพื่อเปิด)</b></summary>

- `zq, _ = qc_frame(Z_raw[i13], noise_dbz=thr)`
- วาดด้วย `mp.radar_map(axs[j], zq, metadata, units="dBZ", title=f"{thr} dBZ", zoom=BOX)`

</details>

In [ ]:
# ✍️ แบบฝึกหัด 0.2 — เขียนโค้ดของคุณที่นี่ (ลบ # หน้าบรรทัดแล้วเติมส่วนที่เป็น ...)
# BOX = tt.range_box(metadata, 165)                     # ซูมรอบรัศมีเรดาร์
# fig, axs = mp.map_panels(1, 3, metadata, panel_width=3.0, zoom=BOX)
# for j, thr in enumerate([10, 15, 20]):
#     zq, _ = qc_frame(Z_raw[i13], noise_dbz=thr)
#     im = mp.radar_map(axs[j], zq, metadata, units="dBZ", title=..., zoom=BOX)
# mp.add_colorbar(fig, im, axs); plt.show()

In [ ]:
#@title ✅ เฉลยแบบฝึกหัด 0.2 (ลองทำเองก่อน! ดับเบิลคลิกเพื่อดูโค้ด แล้วกด ▶ เพื่อรัน)
BOX = tt.range_box(metadata, 165)
fig, axs = mp.map_panels(1, 3, metadata, panel_width=3.0, zoom=BOX)
for j, thr in enumerate([10, 15, 20]):
    zq, fl = qc_frame(Z_raw[i13], noise_dbz=thr)
    im = mp.radar_map(axs[j], zq, metadata, units="dBZ", zoom=BOX, points=False,
                      title=f"Noise threshold {thr} dBZ",
                      gridline_labels=("left", "bottom") if j == 0 else ("bottom",), north=(j == 0))
mp.add_colorbar(fig, im, axs, label="Reflectivity (dBZ)")
mp.panel_labels(axs)
plt.show()
# 10 dBZ: เก็บฝนเบาได้มาก แต่ interference ที่ค่าต่ำหลงเหลือ / 20 dBZ: สะอาดแต่เสียฝนเบา (≈ 0.6 mm/h ด้วย Marshall–Palmer)

### 🧪 แบบฝึกหัด 0.3: หา pixel ของอำเภอเชียงของด้วย affine transform

อำเภอเชียงของอยู่ที่ประมาณ **100.4054°E, 20.2633°N**
1. แปลง lon/lat → UTM 47N ด้วย `pyproj.Transformer`
2. คำนวณ (row, col) ของกริด 500 m ด้วยสมการ affine transform ในข้อ 1.2 (เขียนเอง ห้ามใช้ `tt.xy_to_pixel`)
3. พิมพ์ค่า dBZ (หลัง QC) ที่ pixel นั้นทุกเวลา

**คำถาม:** ช่วงเวลาใดที่เหนือเชียงของมี reflectivity สูงสุด? (คำตอบจะได้ใช้อีกครั้งใน NB1 และ NB5)

<details>
<summary><b>💡 คำใบ้ (คลิกเพื่อเปิด)</b></summary>

- `from pyproj import Transformer`
- `tr = Transformer.from_crs("EPSG:4326", "EPSG:32647", always_xy=True); x, y = tr.transform(lon, lat)`
- จากสมการ: $c = (x - x_0)/\Delta x - 0.5$, $r = (y_0 - y)/\Delta y - 0.5$ โดย $x_0$ = `metadata["x1"]`, $y_0$ = `metadata["y2"]`
- ปัดเป็นจำนวนเต็มด้วย `int(round(...))`

</details>

In [ ]:
# ✍️ แบบฝึกหัด 0.3 — เขียนโค้ดของคุณที่นี่ (ลบ # หน้าบรรทัดแล้วเติมส่วนที่เป็น ...)
# from pyproj import Transformer
# tr = Transformer.from_crs("EPSG:4326", "EPSG:32647", always_xy=True)
# x, y = tr.transform(100.4054, 20.2633)
# col = ...
# row = ...
# print(row, col, Z_qc[:, row, col])

In [ ]:
#@title ✅ เฉลยแบบฝึกหัด 0.3 (ลองทำเองก่อน! ดับเบิลคลิกเพื่อดูโค้ด แล้วกด ▶ เพื่อรัน)
from pyproj import Transformer
tr = Transformer.from_crs("EPSG:4326", "EPSG:32647", always_xy=True)
x, y = tr.transform(100.4054, 20.2633)
col = int(round((x - metadata["x1"]) / metadata["xpixelsize"] - 0.5))
row = int(round((metadata["y2"] - y) / metadata["ypixelsize"] - 0.5))
print(f"UTM: x = {x:.0f} m, y = {y:.0f} m  →  row = {row}, col = {col}")
for t, v in zip(timestamps, Z_qc[:, row, col]):
    print(f"  {mp.tlabel(t)}: {v:5.1f} dBZ")

---
## 📝 สรุป
- GeoTIFF มี **CRS + affine transform** ซึ่งแปลงเป็น `metadata` ของ pysteps ได้โดยตรง
- ต้องแยก **NaN (ไม่มีข้อมูล)** ออกจาก **0 (ไม่มีฝน)** ด้วย range mask
- QC แบบ object-based (ขนาด, รูปร่าง, ทิศทาง) ช่วยลบ speckle และ radial spike และการวิเคราะห์ตามเวลาช่วยหา ground clutter แต่ต้องตรวจด้วยตาเสมอ
- เราได้ข้อมูล 3 ชุด: dBZ 500 m, dBZ 1 km (max) และ rain rate 1 km (mean)

➡️ **บทถัดไป:** Notebook 1 — ดูพายุลูกเห็บเชียงของผ่านสายตาเรดาร์ และแปลง dBZ เป็นปริมาณฝน

### 📚 เอกสารอ้างอิง
- Fabry, F. (2015). *Radar Meteorology: Principles and Practice*. Cambridge University Press. (บทที่ 5–6: data quality)
- Saltikoff, E., et al. (2016). The threat to weather radars by wireless technology. *Bull. Amer. Meteor. Soc.*, 97, 1159–1167.
- ข่าวพายุลูกเห็บเชียงของ: [สำนักงานนโยบายและแผนทรัพยากรธรรมชาติและสิ่งแวดล้อม (สผ.), 24 เม.ย. 2563](https://www.onep.go.th/?p=8239)
- ข้อมูลเรดาร์: กรมอุตุนิยมวิทยา (TMD)